# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://doi.org/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library. We'll load the dataset, review its structure, extract data via Croissant entity `@id`s, explore and process selected fields, and visualize results.

### Dataset Source
The dataset source is available as a Croissant schema:
- URL: https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant metadata and manifest describing the structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
meta_json = metadata.to_json()

# Print basic dataset information
print(f"{meta_json['name']}\n{'='*len(meta_json['name'])}\n{meta_json['description']}")

## 2. Data Overview
Review available record sets, fields, and their Croissant `@id`s. This helps identify the data structure to use in subsequent extraction and analysis steps.

In [ ]:
# Utility to pretty-print JSON
def pretty(obj):
    print(json.dumps(obj, indent=2))

print('Record Sets in Data Package:')
record_sets = metadata.record_sets
for rs in record_sets:
    print(f"- {rs['@id']}: {rs['name']} ({rs['description']})")

# For each record set, show its fields and their @id
for rs in record_sets:
    print(f"\nFields in RecordSet {rs['@id']}: {rs['name']}")
    for f in rs['fields']:
        print(f"  - {f['@id']}: {f.get('name', '')} ({f.get('description','')})")

## 3. Data Extraction
Let's load each record set by its `@id`. We'll extract the first main record set (commonly the survey or main table) into a DataFrame for analysis.

In [ ]:
# Get all record set @id values
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from RecordSet {rs_id}")
    except Exception as ex:
        print(f"Skipping RecordSet {rs_id}: {ex}")

# Select the main survey data RecordSet
# The main record set often has '@id' like 'kilifi_survey' or a similar survey-related id.
# We'll guess it's the first one (else override here).
main_rs_id = record_set_ids[0]
print(f"\nData columns for RecordSet {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
print("Preview:")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic processing: filter numeric records, normalize, group by demographics. We always reference fields by their Croissant `@id` as shown in section 2. Replace `@id` below as needed.

In [ ]:
# Inspect field @id's for valid numeric targets (e.g. GAD-7, PHQ-9, PSQ)
# Here we choose 'gad7_score' and group by, e.g., 'gender' (adapt ids if needed)
numeric_field = 'gad7_score' # Update if another @id is correct
group_field = 'gender' # Use a field present in this dataset
df = dataframes[main_rs_id]

# Only keep rows with non-null and numeric GAD-7
if numeric_field in df.columns:
    filtered = df[df[numeric_field].apply(pd.to_numeric, errors='coerce').notnull()].copy()
    filtered[numeric_field] = filtered[numeric_field].astype(float)

    # Remove outliers (e.g., GAD-7 valid range is 0-21)
    filtered = filtered[(filtered[numeric_field] >= 0) & (filtered[numeric_field] <= 21)]
    # Normalize
    filtered[f"{numeric_field}_normalized"] = (
        (filtered[numeric_field] - filtered[numeric_field].mean()) / filtered[numeric_field].std()
    )

    print(filtered[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field (e.g., gender)
    if group_field in filtered.columns:
        grouped = (
            filtered.groupby(group_field)[numeric_field]
            .agg(['mean', 'std', 'count'])
            .reset_index()
        )
        print(f"\nMean/Std/count of {numeric_field} grouped by {group_field}:")
        print(grouped)
else:
    print(f"{numeric_field} is not found in columns: {df.columns.tolist()}")

## 5. Visualization
Let's visualize the distribution of GAD-7 scores and average by gender.

You can adapt the field `@id`s below for other numeric fields (e.g., PHQ-9, PSQ).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered[numeric_field], bins=22, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field in filtered.columns:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()
else:
    print(f"Column {numeric_field} missing for visualization.")

## 6. Conclusion
In this notebook, we successfully loaded the Kilifi County FAIR² Mental Health Survey dataset via its Croissant schema using `mlcroissant`, explored the available record sets and fields *by their `@id`*, and performed basic exploratory analysis on an assessment score (GAD-7). You can extend this workflow to work with other record sets or fields by updating the `@id` identifiers as needed.

Key findings:
- The dataset supports direct programmatic exploration thanks to the Croissant schema.
- Analysis by fields such as gender or education can reveal distribution differences in mental health scores.
- Use the printed `@id` values to dynamically reference structure in further scripts or notebooks.